# The model comparison (Phase 6, exit test)

This notebook builds the table of the exit test from your result files:

- `evals/results/models/<route>.jsonl`: the test set on each route of the gateway (`make model-testset ROUTE=...`).
- `evals/results/load/<route>.json`: the load test of each open model (`make loadtest ROUTE=...`).

For each route, the table gives the quality (the share of cases with `code_pass`), the latency of a case
(p50 and p95, all model calls of the case), the GPU memory, and the cost of 1,000 model calls. A route with a
load test result is an open model: its cost comes from the hourly price of the GPU and the requests per hour
that you measured. A route without one is a hosted model: its cost comes from its token prices. The cost
functions are yours, in `evals/model_costs.py` (week 25). Until you write them, the cost column stays empty.

The table goes to `evals/results/model_comparison.csv`. Run all cells again after each new result file.

Open it with `make notebook NB=phases/phase-6-open-models/notebooks/model_comparison.ipynb`.

In [ ]:
import csv
import json
import math
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from evals import model_costs  # noqa: E402

RESULTS = ROOT / "evals" / "results"
OUT = RESULTS / "model_comparison.csv"


def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


runs = {path.stem: read_jsonl(path) for path in sorted((RESULTS / "models").glob("*.jsonl"))}
loads = {path.stem: json.loads(path.read_text()) for path in sorted((RESULTS / "load").glob("*.json"))}
print("test set results:", list(runs))
print("load test results:", list(loads))

In [ ]:
def percentile(values, fraction):
    """Nearest rank, as in scripts/loadtest.py."""
    if not values:
        return 0.0
    ordered = sorted(values)
    return ordered[max(1, math.ceil(fraction * len(ordered))) - 1]


def cost_of(route, rows):
    """The cost of 1,000 model calls and its basis, or (None, reason) when it cannot be computed."""
    try:
        if route in loads:
            load = loads[route]
            per_hour = model_costs.best_requests_per_hour(load)
            return model_costs.gpu_cost_per_1000(load["gpu_price_per_hour"], per_hour), "GPU hour"
        return model_costs.hosted_cost_per_1000(rows), "tokens"
    except NotImplementedError:
        return None, "evals/model_costs.py is not written yet"


table = []
for route, rows in runs.items():
    latencies = [row["latency_ms"] for row in rows]
    cost, basis = cost_of(route, rows)
    table.append({
        "model": route,
        "cases": len(rows),
        "quality": round(sum(bool(row["code_pass"]) for row in rows) / len(rows), 3),
        "latency_ms_p50": round(percentile(latencies, 0.5), 1),
        "latency_ms_p95": round(percentile(latencies, 0.95), 1),
        "gpu_memory_gib": loads.get(route, {}).get("gpu_memory_gib"),
        "cost_per_1000_requests": None if cost is None else round(cost, 4),
        "cost_basis": basis,
    })

In [ ]:
with OUT.open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(table[0]))
    writer.writeheader()
    for row in table:
        writer.writerow({key: "" if value is None else value for key, value in row.items()})
print(f"written to {OUT.relative_to(ROOT)}")

try:
    import pandas as pd

    display(pd.DataFrame(table).set_index("model"))
except ImportError:
    for row in table:
        print(row)

Read the table with the error notes of Phase 3 at hand: a lower quality is a list of failed cases, and the
failed cases of the open model tell you which task to fine-tune (week 24). Compare the costs at the load that
your users will really send: an idle GPU still costs its hourly price.